In [ ]:
import logging
from osgeo import gdal, ogr, osr, gdal_array
from stackstac import stack
from rasterio.features import geometry_mask

import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm

import os
import glob
import pandas as pd
import geopandas as gpd
import numpy as np
import shutil
import matplotlib.pyplot as plt
from shapely.geometry import *
from datetime import datetime, date
from dateutil.relativedelta import relativedelta
import math
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import xarray as xr

max_workers=6 # Ajustar de acordo com capacidade do processador

from odc.stac import load
import rioxarray  # Para habilitar .rio

from pystac_client import Client
from odc.stac import load

import warnings
warnings.filterwarnings("ignore")

print(datetime.now())


# Constantes

In [ ]:
##################################################################
# Constantes Hants
feaToMinLimit=  {'ndvi':-0.3, 'savi':-0.3, 'evi':-0.3, 'ndmi':0.0,  'ndwi':0.0   }
feaToMaxLimit=  {'ndvi':1.,   'savi':1.,   'evi':1.,   'ndmi':1.,   'ndwi':1.,   }
feaToErrorLimit={'ndvi':0.1,  'savi':0.1,  'evi':0.1,  'ndmi':0.1,  'ndwi':0.1,  }
feaToOutlier=   {'ndvi':'Lo', 'savi':'Lo', 'evi':'Lo', 'ndmi':'Lo', 'ndwi':'Hi'  }
feaToNodata=    {'ndvi':-3000,'savi':-9999,'evi':-9999,'ndmi':-9999,'ndwi':-9999, 'rgb': 0}

##################################################################




# Funcoes gerais

In [ ]:
def polygonize(input_file, outname):
    shp_name = os.path.basename(outname).split('.')[0]
    command = 'gdal_polygonize.py '+input_file+' -b 1 -f "GeoJSON" '+outname+' '+shp_name+' DN'
    print(command)
    os.system(command)
    

def shapefile2geojson(shp,outname):

    df = gpd.read_file(shp)
    if(os.path.isfile(outname)):
        os.remove(outname)
    df.to_file(outname,driver='GeoJSON')

def polygon_to_json(polygon):
    """
    Converte um polígono Shapely para um dicionário JSON com tipo de geometria e coordenadas.
    
    :param polygon: Objeto Polygon do Shapely.
    :return: Dicionário JSON com formato específico.
    """
    return {
        "type": "Polygon",
        "coordinates": [list(map(list, polygon.exterior.coords))]
    }

def get_total_bounds_polygon(gdf):
    """
    Obtém os limites totais de um GeoDataFrame e cria um polígono a partir deles.
    
    :param gdf: GeoDataFrame de entrada.
    :return: Objeto Polygon representando os limites totais.
    """
    minx, miny, maxx, maxy = gdf.total_bounds
    return Polygon([(minx, miny), (minx, maxy), (maxx, maxy), (maxx, miny), (minx, miny)])


def convert_lonlat_to_utm(lon, lat):
    utm_band = str(int((math.floor((lon+180)/6)%60)+1))
    if len(utm_band) == 1:
        utm_band = '0'+utm_band
    if lat >= 0:
        epsg_code = '326' + utm_band
    else:
        epsg_code = '327' + utm_band
    return epsg_code

def array_2_raster(tiff_path,data_array,gt,prj=None,scale_factor=1.0,nodata=-0.3,type_data='float'):
    if scale_factor!=1.0:
        data_array[data_array!=nodata]*=scale_factor
    ds_scaled=gdal_array.OpenArray(data_array)# convert for gdal object
    ds_scaled.SetGeoTransform(gt)
    if prj is None:
        srs = osr.SpatialReference()
        srs.ImportFromEPSG(4326)
        ds_scaled.SetProjection(srs.ExportToWkt())
    else:    
        ds_scaled.SetProjection(prj)            
    if type_data=='int16':
        gdal_format = gdal.GDT_Int16
    elif type_data=='int32':
        gdal_format = gdal.GDT_Int32
    elif type_data=='float':
        gdal_format = gdal.GDT_Float32
    elif type_data=='uint16':
        gdal_format = gdal.GDT_UInt16
    elif type_data=='bool' or type_data=='uint8':
        gdal_format = gdal.GDT_Byte
    options_gdal=gdal.TranslateOptions(outputType=gdal_format,noData=str(nodata),
                                        creationOptions=["COMPRESS=LZW"])
    gdal.Translate(tiff_path,ds_scaled,outputType=gdal_format,
                    options=options_gdal)

    ds_scaled=None


def raster_2_array(raster_path):
    """
    path_file: it can be a .tif or a .jp2
    """
    raster_ds = gdal.Open(raster_path, gdal.GA_ReadOnly)
    dataset_band = raster_ds.GetRasterBand(1)
    gt=raster_ds.GetGeoTransform()
    dataset_band_array = dataset_band.ReadAsArray()
    prj=raster_ds.GetProjection()
    return dataset_band_array,gt,prj

def stack_tifs_as_arrays(fileslist):
    array,gt,proj=raster_2_array(fileslist[0])
    for i in range(1,len(fileslist)):
        array=np.dstack((array,raster_2_array(fileslist[i])[0]))
    return array,gt,proj


def check_dir(path):
    if(not os.path.isdir(path)):
        os.mkdir(path)

def calcular_area_ha(gdf: gpd.GeoDataFrame, crs_proj='EPSG:3857') -> gpd.GeoDataFrame:

    # Garantir que a geometria seja projetada (metros)
    gdf_proj = gdf.to_crs(crs_proj)
    
    # Calcular área em m² e converter para hectares
    gdf['area_ha'] = gdf_proj.geometry.area / 10000.
    
    return gdf

# Step 0-A = Definir pasta de interesse e datas de execucao

In [ ]:

# Apontar para pasta com shapefile de interesse
path = './input_shapefile/area_teste_13/'
folders = [path]

print(folders)



# Datas de execucao (De um ano atras ate hoje)
# Se for ajustar datas, tentar usar janela de 1 ano - para nao precisar redefinir parametros de filtro temporal

start_date = (datetime.today() - relativedelta(years=1)).strftime('%Y-%m-%d')
end_date = datetime.today().strftime('%Y-%m-%d')

# start_date = '2025-05-01'
# end_date = '2025-05-08'

print("start_date:", start_date)
print("end_date  :", end_date)


# Step 0-B= Buscar serie ndvi de cada poligono e fazer download


## Funcoes STAC

### version_1 - adicionando agrupamento de 10 dias



In [ ]:
def get_ndvi_series_stac(geo_name, path_out_fea, feature, start_date, end_date, fnodata=-9999,period_days=5,max_workers=4):

    total_initime = datetime.now()
    
    # Load geojson
    df_field = gpd.read_file(geo_name)

    # Converter em json para query
    bounds = get_total_bounds_polygon(df_field)
    geometry_test  = polygon_to_json(bounds)
    ##############################################################
    # use publically available stac link such as

    url = "https://earth-search.aws.element84.com/v1"
    # url = "https://planetarycomputer.microsoft.com/api/stac/v1"


    print(f'Open url - {url}')
    client = Client.open(url) 
    
    # ID of the collection
    collection = "sentinel-2-l2a"
    ##############################################################

    print('# 1. STAC: busca dos itens Sentinel-2 L2A')

    initime = datetime.now()

    # run pystac client search to see available dataset
    search = client.search(
        collections=[collection], intersects=geometry_test, datetime=start_date+'/'+end_date
    )
    print('search ok', len(search.item_collection_as_dict()) )
    print('step 1 - elapsed time =' , datetime.now() - initime )
    ##############################################################
    print('# 2. Carregar dados via stack')
    initime = datetime.now()
    
    data = load(search.items() ,geopolygon=geometry_test,groupby="solar_day", chunks={})
    
    print('elapsed time =' , datetime.now() - initime )
    ##############################################################
    print('# 3. Calcular NDVI')
    initime = datetime.now()
    
    # create the index without considering scale or offset
    ndvi = (data.nir - data.red) / (data.nir + data.red)
    # Limitar a -1 e 1
    ndvi = ndvi.where((data.nir + data.red) != 0)  # evitar divisão por zero
    ndvi = ndvi.where((ndvi >= feaToMinLimit[feature]) & (ndvi <= feaToMaxLimit[feature]), 0)
    # Aplicar fator de escala 10k para usar int16
    ndvi = (ndvi * 10000).round().astype("int16")
    
    print('elapsed time =' , datetime.now() - initime )
    ##############################################################
    print('# 4. Agrupar por período e exportar NDVI')
    initime = datetime.now()
    
    # # Reprojetar o shapefile para a mesma projeção do raster
    shape = df_field.to_crs(ndvi.rio.crs)
    print('shape_area_ha =',shape.geometry.area.sum() / 10000.) 
    
    # Obter as datas de cada imagem
    dates = pd.to_datetime(ndvi.time.values)
    
    # Adicionar uma coluna de "período" (5 dias ou 10 dias)
    period_start = pd.to_datetime(start_date)
    period_end = pd.to_datetime(end_date)
    
    def get_period(date, period_days):
        # Calcula o número de dias desde a data inicial e retorna o número do período
        delta_days = (date - period_start).days
        period_number = delta_days // period_days
        return period_start + pd.to_timedelta(period_number * period_days, 'D')
    
    # Aplicar a função para categorizar as datas em períodos
    period_dates = dates.map(lambda x: get_period(x, period_days))
    
    print('Criar uma nova dimensão para os períodos...')
    print('>>',datetime.now() - initime)
    ndvi['period'] = ('time', period_dates)
    print('>>',datetime.now() - initime)
    
    
    # Agrupar pelo período e pegar o valor máximo de NDVI
    ndvi_grouped = ndvi.groupby('period').max(dim='time')

    def export_ndvi_period(period_idx):
        period_start_date = str(ndvi_grouped.period.values[period_idx])[:10]
        out_path = os.path.join(path_out_fea, f"ndvi_{period_start_date}_max{period_days}D.tif")
        if not os.path.isfile(out_path):
            ndvi_period = ndvi_grouped.isel(period=period_idx)
    
            # Clipping com shapefile (já no CRS do raster original)
            ndvi_period = ndvi_period.rio.clip(shape.geometry, shape.crs, all_touched=False, drop=True)
    
            # Reprojetar para EPSG:4326 antes de salvar
            ndvi_period = ndvi_period.rio.reproject("EPSG:4326")
    
            # Aplicar o valor nodata para as áreas fora da geometria
            ndvi_period = ndvi_period.rio.write_nodata(fnodata)
    
            # Exportar GeoTIFF
            ndvi_period.rio.to_raster(out_path, compress="LZW", tiled=True, dtype="int16")

        # print(f"Salvo: {out_path}")

    # Exportar os arquivos para cada período
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(export_ndvi_period, i) for i in range(len(ndvi_grouped.period))]
    
        for _ in tqdm(as_completed(futures), total=len(futures), desc="Exportando NDVI agrupado"):
            pass
    
    print("Exportação finalizada.")
    print('elapsed time =' , datetime.now() - initime )
    print('Total elapsed time =' , datetime.now() - total_initime )
    print('=====================================')

print(f'ok - {datetime.now()}')

## Get NDVI series

In [ ]:
feature = 'ndvi'

period_days = 10 # numero de dias para agrupar o NDVI maximo

max_workers= 6 # Ajustar de acordo com capacidade do processador

for folder in folders:
    print(folder)
    # Buscar shapefile de interesse
    find = sorted(glob.glob(os.path.join(folder,'*.shp')))
    print(find)

    

    # Plotar poligono temp
    tempdf= gpd.read_file(find[0])
    tempdf = calcular_area_ha(tempdf)
    display(tempdf.head())
    
    tempdf.plot(facecolor='none',edgecolor='blue',linewidth=2.)
    plt.show()

    # Diretorios da feature - NDVI
    path_out_fea = os.path.join(folder,feature)
    print('path_out_fea',path_out_fea)
    check_dir(path_out_fea)

    # Somente NDVI
    get_ndvi_series_stac(find[0], path_out_fea,
                         feature, start_date, end_date,
                         fnodata=feaToNodata[feature],
                         period_days=period_days,
                         max_workers=max_workers
                        )




# Get RGB series (Opcional)

In [ ]:
def get_rgb_series_stac(geo_name, path_out_rgb, start_date, end_date, fnodata=0, period_days=5, max_workers=4):
    total_initime = datetime.now()
    
    # Load geojson
    df_field = gpd.read_file(geo_name)

    # Converter em json para query
    bounds = get_total_bounds_polygon(df_field)
    geometry_test  = polygon_to_json(bounds)
    ##############################################################
    # use publically available stac link such as

    url = "https://earth-search.aws.element84.com/v1"
    # url = "https://planetarycomputer.microsoft.com/api/stac/v1"


    print(f'Open url - {url}')
    client = Client.open(url) 
    
    # ID of the collection
    collection = "sentinel-2-l2a"
    ##############################################################

    print('# 1. STAC: busca dos itens Sentinel-2 L2A')

    initime = datetime.now()

    # run pystac client search to see available dataset
    search = client.search(
        collections=[collection], intersects=geometry_test, datetime=start_date+'/'+end_date
    )
    print('search ok', len(search.item_collection_as_dict()) )
    print('step 1 - elapsed time =' , datetime.now() - initime )
    ##############################################################

     ##############################################################
    print('# 2. Carregar dados via stack')
    initime = datetime.now()
    
    data = load(search.items() ,geopolygon=geometry_test,groupby="solar_day", chunks={})
    
    print('elapsed time =' , datetime.now() - initime )
    ##############################################################

    print('# 3. Selecionar bandas para composição RGB (Falsa cor)')
    band_r = data.nir  # NIR como vermelho
    band_g = data.swir16  # SWIR B11 como verde
    band_b = data.red  # RED como azul

    print('# 4. Agrupar por período e exportar imagens RGB')
    shape = df_field.to_crs(data.rio.crs)
    dates = pd.to_datetime(data.time.values)
    period_start = pd.to_datetime(start_date)

    def get_period(date):
        delta_days = (date - period_start).days
        period_number = delta_days // period_days
        return period_start + pd.to_timedelta(period_number * period_days, 'D')

    period_dates = dates.map(get_period)
    band_r['period'] = ('time', period_dates)
    band_g['period'] = ('time', period_dates)
    band_b['period'] = ('time', period_dates)

    band_r_grouped = band_r.groupby('period').mean(dim='time')
    band_g_grouped = band_g.groupby('period').mean(dim='time')
    band_b_grouped = band_b.groupby('period').mean(dim='time')


    def export_rgb_period(period_idx):
        period_start_date = str(band_r_grouped.period.values[period_idx])[:10]
        out_path = os.path.join(path_out_rgb, f"rgb_{period_start_date}_mean{period_days}D.tif")
        # print(f"Exportando período {period_idx}: {out_path}")
    
        if not os.path.isfile(out_path):
            try:
                r = band_r_grouped.isel(period=period_idx)
                g = band_g_grouped.isel(period=period_idx)
                b = band_b_grouped.isel(period=period_idx)
                rgb = xr.concat([r, g, b], dim='band')
                rgb['band'] = [1, 2, 3]
                
                # print(f"Antes do clip: shape={rgb.shape}, CRS={rgb.rio.crs}")
                rgb = rgb.rio.clip(shape.geometry, shape.crs, all_touched=False, drop=True)
                # print(f"Depois do clip: shape={rgb.shape}")
                
                # rgb = rgb.rio.reproject("EPSG:4326") # Reprojetar é desnecessario aqui (somente visualizacao)
                rgb = rgb.rio.write_nodata(fnodata)
                rgb.rio.to_raster(out_path, compress="LZW", tiled=True, dtype="uint16",nodata=fnodata)
                # print(f"Salvo: {out_path}")
            except Exception as e:
                print(f"Erro ao exportar {out_path}: {e}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(export_rgb_period, i) for i in range(len(band_r_grouped.period))]
        for _ in tqdm(as_completed(futures), total=len(futures), desc="Exportando RGB agrupado"):
            pass

    print("Exportação finalizada.")
    print('Total elapsed time =', datetime.now() - total_initime)
    print('=====================================')


In [ ]:
feature2 = 'rgb'

period_days = 5 # numero de dias para agrupar o NDVI maximo


start_date2= '2024-09-01'

for folder in folders:
    print(folder)
    # Buscar shapefile de interesse
    find = sorted(glob.glob(os.path.join(folder,'*.shp')))
    print(find)

    

    # Plotar poligono temp
    tempdf= gpd.read_file(find[0])
    tempdf = calcular_area_ha(tempdf)
    display(tempdf.head())

    print('AREA_HA=',tempdf.area_ha.sum())
    tempdf.plot(facecolor='none',edgecolor='blue',linewidth=2.)
    plt.show()

    # Diretorios da feature - NDVI
    path_out_rgb = os.path.join(folder,feature2)
    print('path_out_rgb',path_out_rgb)
    check_dir(path_out_rgb)

    # Download RGB
    get_rgb_series_stac( 
        find[0], path_out_rgb, start_date2, end_date, fnodata=0, period_days=10, max_workers=4)


# Step 1 - Filtrar serie temporal com Hants

## Funcoes Hants ok


In [ ]:
def check_dir(path_check):
    if not os.path.exists(path_check):
        os.makedirs(path_check)

def array_in(y):
    y = np.tile(y[None].T, (1, 1)).T
    kl=np.ones([1,1])
    y = y * kl
    return y

def create_virtual_raster(flist,outname, srcnodata=0):

    flist_str=''
    for i in range(len(flist)):
        flist_str=flist_str+' '+flist[i]
        
    os.system('gdalbuildvrt -separate -srcnodata {} {} {}'.format(srcnodata,os.path.abspath(outname),flist_str))
    return outname

def get_starter_matrix(base_period_len, sample_count, frequencies_considered_count):
    nr = min(2 * frequencies_considered_count + 1,
                sample_count)  # number of 2*+1 frequencies, or number of input images
    mat = np.zeros(shape=(nr, sample_count))
    mat[0, :] = 1
    ang = 2 * np.pi * np.arange(base_period_len) / base_period_len
    cs = np.cos(ang)
    sn = np.sin(ang)
    # create some standard sinus and cosinus functions and put in matrix
    i = np.arange(1, frequencies_considered_count + 1)
    ts = np.arange(sample_count)
    for column in range(sample_count):
        index = np.mod(i * ts[column], base_period_len)
        # index looks like 000, 123, 246, etc, until it wraps around (for len(i)==3)
        mat[2 * i - 1, column] = cs.take(index)
        mat[2 * i, column] = sn.take(index)
    return mat

def makediag4d(M):

    N = np.ones((M.shape[0], M.shape[1], M.shape[-1], M.shape[-1]))

    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            
            N[i,j] = np.diag(M[i,j,:])

    return N


def HANTS(inputs, mat,
        outliers_to_reject='Lo',low=0., high=255,
        fit_error_tolerance=5,delta=0.1):
    """
    Function to apply the Harmonic analysis of time series applied to arrays
    sample_count    = nr. of images (total number of actual samples of the time series)
    base_period_len    = length of the base period, measured in virtual samples
            (days, dekads, months, etc.)
    frequencies_considered_count    = number of frequencies to be considered above the zero frequency
    inputs     = array of input sample values (e.g. NDVI values)
    ts    = array of size sample_count of time sample indicators
            (indicates virtual sample number relative to the base period);
            numbers in array ts maybe greater than base_period_len
            If no aux file is used (no time samples), we assume ts(i)= i,
            where i=1, ..., sample_count
    outliers_to_reject  = 2-character string indicating rejection of high or low outliers
            select from 'Hi', 'Lo' or 'None'
    low   = valid range minimum
    high  = valid range maximum (values outside the valid range are rejeced
            right away)
    fit_error_tolerance   = fit error tolerance (points deviating more than fit_error_tolerance from curve
            fit are rejected)
    dod   = degree of overdeterminedness (iteration stops if number of
            points reaches the minimum required for curve fitting, plus
            dod). This is a safety measure
    delta = small positive number (e.g. 0.1) to suppress high amplitudes
    """
    
    try:
        sample_count = inputs.shape[-1]

        # check which setting to set for outlier filtering
        if outliers_to_reject == 'Hi':
            sHiLo = -1
        elif outliers_to_reject == 'Lo':
            sHiLo = 1
        else:
            sHiLo = 0
        nr = min(2 * freq + 1,
                sample_count)  # number of 2*+1 frequencies, or number of input images

        # create empty arrays to fill
        outputs = np.zeros(shape=(inputs.shape[0], inputs.shape[1], sample_count))

        p = np.ones_like(inputs)
        p[(low > inputs) | (inputs > high)] = 0
        nout = np.sum(p == 0, axis=-1)  # count the outliers for each timeseries
        
        # prepare for while loop
        ready = np.zeros((inputs.shape[0], inputs.shape[1]), dtype=bool)  # all timeseries set to false
        dod = 1  # (2*frequencies_considered_count-1)  # Um, no it isn't :/
        noutmax = sample_count - nr - dod
        
        for _ in range(sample_count):
            if ready.all():
                break

            # multiply outliers with timeseries
            za = np.einsum('mk,ijk->ijm', mat, p * inputs)
            
            # multiply mat with the multiplication of multiply diagonal of p with transpose of mat
            diag = makediag4d(p)
            A = np.einsum('kl,ijlm->ijkm', mat, np.einsum('ijkl,lm->ijlm', diag, mat.T))

            # add delta to suppress high amplitudes but not for [0,0]
            A = A + np.tile(np.expand_dims(np.diag(np.ones(15)), axis=(0,1)), (inputs.shape[0], inputs.shape[1], 1, 1)) * delta
            A[:, :, 0, 0] = A[:, :, 0, 0] - delta
            
            # solve linear matrix equation and define reconstructed timeseries
            zr = np.linalg.solve(A, za)
            outputs = np.einsum('km,ijm->ijk', mat.T, zr)

            # calculate error and sort err by index
            err = p * (sHiLo * (outputs - inputs))
            rankVec = np.argsort(err, axis=-1, )
            
            # select maximum error and compute new ready status
            maxerr = np.max(err, axis=-1)

            ready = (maxerr <= fit_error_tolerance) | (nout == noutmax)

            # if ready is still false
            if not ready.all():
                j = rankVec[:,:,sample_count-1]

                p[np.indices(j.shape)[0], np.indices(j.shape)[1], j] = p[np.indices(j.shape)[0], np.indices(j.shape)[1], j] * ready.astype(int)#*check
                
                nout += 1

        return outputs,zr

    except Exception as e:
        _, _, exc_tb = sys.exc_info()
        raise Exception(f'[ERROR - HANTSOPT]: {e} | Line: {exc_tb.tb_lineno}')


def call_hants_parallel( flist,path_hants,year,fea,freq_recon_ndvi,mat,vrt_recon,nodata=0):
    ## Criar diretorio de saida se nao existir
    check_dir(path_hants)

    
    #==========================================================================
    #loading data
    array,gt,proj=stack_tifs_as_arrays(fileslist=flist)
    
    array=array.astype(float)

    nodata_mask = np.where(array==feaToNodata[feature])

    array=array/10000.0
    
    if 'ndmi'==fea or 'ndwi'==fea:
        scaler = MinMaxScaler(feature_range=(0, 10000.0))
        for k in range(array.shape[2]):
            array[:,:,k]=scaler.fit_transform(array[:,:,k])
    else:            
        array[array<feaToMinLimit[fea]]=feaToMinLimit[fea]

    #==========================================================================
    # appling hants 

    try:
        _, zr_arr = HANTS(inputs=array, mat=mat,
                        outliers_to_reject=feaToOutlier[fea], low=feaToMinLimit[fea], high=1.0, fit_error_tolerance=feaToErrorLimit[fea])

        array_shape=array.shape
        array=None

        #==========================================================================
        # saving results - amp and phase
        # Initializing amp and phase arrays:
        amp=np.zeros([array_shape[0],array_shape[1],freq+1],dtype=float)
        phase=np.zeros([array_shape[0],array_shape[1],freq+1],dtype=float)

        # Storing the amplitude for 0 rad frequency:
        amp[:,:,0]=zr_arr[:,:,0]
        
        freq_recon = freq_recon_ndvi if fea == 'ndvi' else freq_recon_reflectance
        # Managing dirs and paths:
        path_feature_hants=os.path.join(path_hants,'amp_pha_'+str(freq)+'freq')
        path_feature_hants_res=os.path.join(path_hants,'reconstructed_'+str(freq_recon)+'freq')
        check_dir(path_feature_hants)
        check_dir(path_feature_hants_res)

        # save amp0 and ph0 -> phase 0  is null
        outname=os.path.join(path_feature_hants,'hants_am0_'+fea+'_'+year+'.tif')
        _ = array_2_raster(outname,amp[:,:,0],gt,proj)
        outname=os.path.join(path_feature_hants,'hants_ph0_'+fea+'_'+year+'.tif')
        _ = array_2_raster(outname,phase[:,:,0],gt,proj)

        # save N frequencies
        for f in range(1,2*freq+1,2):
            i=int(((f-1)/2)+1) # freq number
            
            zr_real=zr_arr[:,:,f]
            zr_img=zr_arr[:,:,f+1]
            
            amp[:,:,i]=(zr_real**2+zr_img**2)**0.5

            # TODO: Change these for loops for a matrix approach (np.angle)
            for m in range(amp.shape[0]):
                for n in range(amp.shape[1]):
                    if(zr_real[m,n]==0):
                        phase[m,n,i]=(np.pi/2)*180.0/np.pi
                    elif(zr_real[m,n]<0):
                        phase[m,n,i]=(np.arctan(zr_img[m,n]/zr_real[m,n])+np.pi)*(180.0/np.pi)
                    else:
                        phase[m,n,i]=(np.arctan(zr_img[m,n]/zr_real[m,n]))*(180.0/np.pi)

                    if(phase[m,n,i]<0):
                        phase[m,n,i]=phase[m,n,i]+360.0
                        
            outname=os.path.join(path_feature_hants,'hants_am'+str(i)+'_'+fea+'_'+year+'.tif')
            _ = array_2_raster(outname,amp[:,:,i],gt,proj)
            outname=os.path.join(path_feature_hants,'hants_ph'+str(i)+'_'+fea+'_'+year+'.tif')
            _ = array_2_raster(outname,phase[:,:,i],gt,proj)
            
        zr_arr=None
        
        #==========================================================================
        # recompose time series
        if(recon==True):
            yrecon=np.zeros(array_shape)

            # AMP and PHASE resized to freq_recon+1 (axis 2) (select only desired frequencies)
            amp=amp[:,:,0:freq_recon+1]
            phase=phase[:,:,0:freq_recon+1]

            for i in range(yrecon.shape[0]):
                for j in range(yrecon.shape[1]):

                    # TODO: review this line
                    recon_matrix = (np.tile(amp[i,j,:],(yrecon.shape[2], 1)).T*np.cos((np.tile(np.arange(yrecon.shape[2]),(freq_recon+1, 1))*np.arange(freq_recon+1).reshape((-1,1))*2.0*np.pi/yrecon.shape[2])-np.tile(phase[i,j,:]*np.pi/180.0,(yrecon.shape[2],1)).T)).sum(axis=0)

                    yrecon[i,j,:] = recon_matrix

            amp=None
            phase=None
            #==========================================================================
            # saving recomposition of time series
            yrecon[yrecon>1]=1
            yrecon[yrecon<feaToMinLimit[fea]]=feaToMinLimit[fea]

            yrecon[nodata_mask] = feaToNodata[feature] / 10000.

            vrt_list=[]
            for z in range(yrecon.shape[2]):

                outname=os.path.join(path_feature_hants_res,os.path.basename(flist[z].replace('.tif','_hantsrecon_'+str(freq_recon)+'freq.tif')))
                vrt_list.append(outname)
                _ = array_2_raster(outname,yrecon[:,:,z],gt,proj,scale_factor = 10000, nodata=feaToNodata[feature],type_data='int16')

            yrecon=None

            vrt_recon=vrt_recon#os.path.join(path_hants,fea+'_'+year+'_hants_'+str(freq_recon)+'freq.vrt')
            vrt=create_virtual_raster(vrt_list,vrt_recon, srcnodata=feaToNodata[feature])
    
    except Exception as e:
        _, _, exc_tb = sys.exc_info()
        print(f'[ERROR - call_hants_parallel HANTS]: {e} | Line: {exc_tb.tb_lineno}')

## Definir parametros Hants - Nao 'e necessario alterar *

### * aterar somente se quiser range de datas diferentes, desde que tenha sido feito download para pasta local

In [ ]:



#### FREQUENCIES CONSIDERED
# Freqencias para rodar hants e 
freq = 7
freq_recon_reflectance = 4
freq_recon = 4


recon=True # Reconstruct time series in call hants function
feature='ndvi'

years_dates={
'2023':['2022-09-01','2023-08-31'],
'2024':['2023-09-01','2024-08-31'],
'2025':[start_date,end_date], # Ajustar data de inicio e final antes do download dos dados, acima
}

# Ano que vai rodar a filtragem temporal - verificar datas no dicionario acima
year = '2025'


date_start=years_dates[year][0]
date_end  =years_dates[year][1]

date_range=pd.date_range(start=date_start,end=date_end)
date_range=[str(item.date()) for item in date_range]


print('folders=',folders)
print('feature=', feature)
print ('year=',year)
print('freq_recon=',freq_recon)
print ('*** date_start=',date_start)
print ('*** date_end=  ',date_end)



## Aplicar Hants na serie do poligono

In [ ]:

hants_initime = datetime.now()

for folder in folders[0:1]:
    print(folder)


    flist = []
    for dd in date_range:
        # print(dd)
        f=sorted(glob.glob(os.path.join(folder,feature,'*'+dd+'*.tif')))
        if(len(f)>0):
            
            for item in f:
                # print(item)
                flist.append(item)

    print(len(flist),'files')

    print(flist[0])
    print(flist[-1])



    # Criar pasta de output
    path_hants = os.path.join(folder,feature+'_hants_'+year)
    check_dir(path_hants)

    # ####################################

    # Criar vrt da serie oritinal
    vrt_original=os.path.join(folder,f'{feature}_{year}_original_{freq_recon}freq.vrt')
    print([os.path.isfile(vrt_original)],vrt_original)
    create_virtual_raster(flist,vrt_original, srcnodata=feaToNodata[feature])


    vrt_recon=os.path.join(folder,f'{feature}_{year}_hants_{freq_recon}freq.vrt')
    print([os.path.isfile(vrt_recon)],vrt_recon)

    sample_count = len(flist)
    mat = get_starter_matrix(sample_count, sample_count, freq)
    if(not os.path.isfile(vrt_recon)):
        print('run hants!')
        call_hants_parallel(flist,path_hants,year,feature,freq_recon,mat,vrt_recon,nodata=feaToNodata[feature])

    print('Hants elapsed time =', datetime.now() - hants_initime)
    print("FINISHED!")

    

# Clusterizacao - Agrupar pixels com mesmo comportamento em grupos

## Objetivo =  de separar dinamicas
## Resultados = sao salvos em uma subpasta no local do shapefile
### * curvas de ndvi para cada dinamica (cluster)
### * arquivo geotiff (em EPSG:4326)
### * geojson do "polygonize" do geotiff de clusters

## Funcoes cluster

In [ ]:

from sklearn.preprocessing import MinMaxScaler


sizeF=16 # parametros para figura
sizeL=4  # parametros para figura

def filter_images_by_date(flist, start_date_str):
    # Converte a data de corte para datetime
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")

    # Filtra a lista
    filtered_list = [
        f for f in flist 
        if datetime.strptime(os.path.basename(f).split('_')[1], "%Y-%m-%d") >= start_date
    ]
    return filtered_list

import numpy as np
import matplotlib.pyplot as plt

def plot_n_series(series, dates, ylabel, title='', filename=None, colors=None,
                  skip_date=1, show_mean_std=False, mean_color='tab:blue',pltshow=True):
    """
    Plota várias séries temporais ou apenas a média com desvio padrão sombreado.

    Parâmetros:
    - series: numpy array (n_tempos, n_amostras)
    - dates: lista de datas para o eixo X.
    - ylabel: label do eixo Y.
    - title: título do gráfico.
    - filename: se definido, salva o arquivo com este nome.
    - colors: lista de cores (usado apenas se show_mean_std=False).
    - skip_date: define o espaçamento entre labels do eixo X.
    - show_mean_std: se True, plota média e desvio padrão ao invés de todas as curvas.
    - mean_color: cor da linha de média e da faixa de desvio padrão.
    """

    fig = plt.figure(figsize=(10, 4))
    x = np.arange(series.shape[0])

    
    
    if show_mean_std:
        mean_series = np.nanmean(series, axis=1)
        std_series = np.nanstd(series, axis=1)

        # Plotar todas as curvas finas
        for i in range(series.shape[1]):
            plt.plot(x, series[:, i], linewidth=1., ls='-', color='grey', alpha=.2)  # Linha fina e cinza
            # plt.plot(x, series[:, i], linewidth=1., ls='-', alpha=.5)  # Linha fina e cinza
        
        # Plotando a média e a faixa de desvio padrão
        plt.plot(x, mean_series, linewidth=2.5, color=mean_color, label='mean')
        plt.fill_between(x, mean_series - std_series, mean_series + std_series, 
                         color=mean_color, alpha=0.5, label='std')
    else:  
        for i in range(series.shape[1]):
            plt.plot(x, series[:, i], linewidth=1.5, ls='-',alpha=.5, color=colors[i] if colors else None)
        mean_series = np.nanmean(series, axis=1)
        plt.plot(x, mean_series, linewidth=3, ls='-', color='black', label='mean')
        
    plt.legend()
    
    plt.ylabel(ylabel, fontsize=12)

    # gap = max(int(series.shape[0] / skip_date), 1)
    # plt.xticks(x[::gap], dates[::gap], fontsize=10, rotation=90)
    plt.xticks(x, dates,fontsize = 0.75*sizeF-sizeL, rotation=90)   
    plt.yticks(fontsize=10)

    axes = plt.gca()
    axes.set_xlim([-1, series.shape[0]])
    axes.set_ylim([0,10000
        # series.min() - abs(series.min() * 0.01),
        # series.max() + abs(series.max() * 0.01)
    ])

    plt.title(title, fontsize=14)

    if filename is not None:
        format_fig = filename.split('.')[-1]
        plt.savefig(filename, format=format_fig, dpi=100, bbox_inches='tight')

    if(pltshow==True):
         plt.show()
    plt.close()
    

    
def generate_color_dict(k, cmap_name='jet_r'):

    cmap = plt.get_cmap(cmap_name)
    color_dict = {}

    for i in range(k):
        rgba = cmap(i / max(k - 1, 1))  # evita divisão por zero
        hex_color = '#{:02x}{:02x}{:02x}'.format(
            int(rgba[0] * 255),
            int(rgba[1] * 255),
            int(rgba[2] * 255)
        )
        color_dict[i] = hex_color

    return color_dict

def plot_classified_raster(array, geotransform, k2color, title='',nodata=None, figname='classified_raster.png'):
    """
    Plota um raster classificado com base em um dicionário de cores e salva em uma pasta padrão com nome personalizado.

    Parâmetros:
    array (numpy.ndarray): Matriz do raster classificado.
    geotransform (tuple): Geotransform (minX, pixelWidth, 0, maxY, 0, -pixelHeight).
    k2color (dict): Dicionário {classe: cor_hex}.
    nodata (int, opcional): Valor que será tratado como transparente.
    figname (str): Nome do arquivo da figura (ex: 'meu_raster.png').
    path_figs (str, opcional): Caminho da pasta para salvar o plot. O padrão é './figs'.
    """
    class_keys = sorted(k2color.keys())
    color_list = [k2color[k] for k in class_keys]

    cmap = ListedColormap(color_list)

    mask = (array == nodata) if nodata is not None else np.full(array.shape, False)

    xmin = geotransform[0]
    xmax = xmin + array.shape[1] * geotransform[1]
    ymax = geotransform[3]
    ymin = ymax + array.shape[0] * geotransform[5]  # geotransform[5] é negativo

    fig, ax = plt.subplots(figsize=(8, 8))
    img = ax.imshow(
        np.ma.masked_array(array, mask),
        cmap=cmap,
        extent=[xmin, xmax, ymin, ymax],
        interpolation='nearest'
    )

    ax.set_title(title)
    ax.set_xlabel('longitude')
    ax.set_ylabel('latitude')

    handles = [
        plt.Line2D([0], [0], marker='s', color='w', markerfacecolor=k2color[k], markersize=10, label=str(k))
        for k in class_keys
    ]
    if(len(np.unique(array))<20):
        ax.legend(
            handles=handles,
            title='Classes',
            bbox_to_anchor=(1.05, 1),
            loc='upper left'
        )

    plt.tight_layout()

    # Salva o arquivo na pasta padrão (./figs) com o nome fornecido

    save_path = figname#os.path.join(path_figs, figname)
    fig.savefig(save_path, dpi=100, bbox_inches='tight')
    print(f'Figura salva em: {save_path}')

    plt.show()

def cluster_prop(cluster_raster, k_cluster, nodata=-1):

    valid_mask = cluster_raster != nodata
    total = np.sum(valid_mask)

    if total == 0:
        return {k: 0.0 for k in range(k_cluster)}  # Evita divisão por zero.

    count = {}
    for k in range(k_cluster):
        proportion = 100 * np.sum((cluster_raster == k) & valid_mask) / total
        count[k] = round(proportion, 2)
    return count


def raster_to_polygons_with_simplify_fast(raster_path, size=3, nodata=-99, simplify_tolerance_m=10,
                                          out_crs="EPSG:4326",save_filtered_raster=False,filtered_raster_path=None):

    
    """
    Funcao que aplica filtro de mediana
    Poligoniza em geodataframe e retorna gdf + raster filtrado """
    import numpy as np
    import rasterio
    import rasterio.features
    import geopandas as gpd
    from shapely.geometry import shape
    from scipy.ndimage import median_filter

    # Abrir o raster
    with rasterio.open(raster_path) as src:
        raster = src.read(1)
        transform_raster = src.transform
        crs = src.crs
        profile = src.profile

    # Substituir nodata por np.nan temporariamente para evitar contaminar a mediana
    raster_nan = np.where(raster == nodata, np.nan, raster)

    # Aplicar filtro de mediana ignorando NaNs
    raster_filled = np.nan_to_num(raster_nan, nan=nodata)  # usar nodata como preenchimento temporário
    filtered = median_filter(raster_filled, size=size)

    # Restaurar nodata
    filtered[raster == nodata] = nodata

    # Salvar o raster filtrado, se solicitado
    if save_filtered_raster:
        if filtered_raster_path is None:
            filtered_raster_path = raster_path.replace(".tif", "_filtered.tif")
        
        with rasterio.open(filtered_raster_path, "w", **profile) as dst:
            dst.write(filtered.astype(profile["dtype"]), 1)

        print(f">> Raster filtrado salvo em: {filtered_raster_path}")

    # Criar geometria a partir dos valores
    mask = filtered != nodata
    shapes = rasterio.features.shapes(filtered, mask=mask, transform=transform_raster)

    records = [{"geometry": shape(geom), "value": int(value)} for geom, value in shapes if value != nodata]
    gdf = gpd.GeoDataFrame(records, crs=crs)

    # Converter para UTM automaticamente para aplicar simplificação em metros
    utm_crs = gdf.estimate_utm_crs()
    gdf_utm = gdf.to_crs(utm_crs)

    # Aplicar simplificação
    gdf_utm["geometry"] = gdf_utm.geometry.simplify(tolerance=simplify_tolerance_m, preserve_topology=True)

    # Voltar ao CRS de saída
    gdf_out = gdf_utm.to_crs(out_crs)

    return gdf_out, filtered

# CLUSTER 1 - KMeans

## Vantagem - nao interpreta series como ruido >> todo pixel 'e atribuido a um grupo'
## Desvantagem - necessita de input do numero de grupos desejado


## Clusterizacao usando Kmeans

### k_cluster corresponde ao numero de grupos desejado. Sugestao - ao testar uma nova area, considerar 1 classe para vegetacao natural (ruido), e mais uma classe para cada talhao - testar 5 ou 7 grupos inicialmente, e reduzir numero ate obter grupos homogeneos

In [ ]:
# Data para restringir a serie temporal (coloquei inicio da janela do ano safra 2024-25)
# date_slice = start_date

date_slice='2024-09-01'

feature = 'ndvi'

In [ ]:

from sklearn.cluster import KMeans

k_initime = datetime.now()

# Numero de clusters
k_cluster = 5

# Plot de curvas
plot_curves = False # salvar em png
pltshow = False # exibir no jupyter-notebook

 
# Nao alterar esse valor - 'e um parametro da filtragem do ndvi necessaria para encontrar os arquivos
freq_recon= 4




# Cria dicionario de cores para plot (em funcao do numero de grupos desejado)
k2color=generate_color_dict(k_cluster,cmap_name='viridis')



for folder in folders[0:1]:
    print(folder)

    print('find folder=',os.path.join(folder,
                                                feature+'_hants_'+year,
                                                'reconstructed_'+str(freq_recon)+'freq','*.tif'))
    flist = sorted(glob.glob(os.path.join(folder,feature+'_hants_'+year,'reconstructed_'+str(freq_recon)+'freq','*.tif')))
    print(len(flist))

    flist = filter_images_by_date(flist,date_slice)
    print(os.path.basename(flist[0]))
    print(os.path.basename(flist[-1]))    

    # Importar serie temporal NDVI
    arr,gt,proj = stack_tifs_as_arrays(flist)

    # Criar mascara com valores validos da Feature
    arr_mask = np.zeros(arr.shape[0:2])
    arr_mask[np.mean(arr,axis=2)>0] = 100

    ########################## 
    # Cluster
    ########################## 
 

    ###############################xxxxxxxxxxxxxxxxxxxxxx
    # Cria máscara de pixels válidos (média maior que zero)

    arr_mask = np.mean(arr, axis=2) > 0
    idx = np.where(arr_mask >0) # mascara de pixels de interesse
    
    # Extrai apenas os pixels válidos
    X_origin = arr[arr_mask]
    
    # Normalização Min-Max por pixel (cada série de tempo é reescalada)
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X_origin)
    
    # Clusterização com KMeans
    time_ini = datetime.now()
    kmeans = KMeans(n_clusters=k_cluster, random_state=0).fit(X_scaled)
    labels = kmeans.fit_predict(X_scaled)
    time_end = datetime.now()
    print(f"Clustering finalizado em: {time_end - time_ini}")
    ##########################################xxxxxxxxxxxxxxx


    #############################################
    # Opcional - Gerar pasta para salvar figuras dos clusters
    #############################################

    path_figs=os.path.join(folder,f'{feature}_hants{freq_recon}_{year}_figures_cluster_Kmeans_k{k_cluster}')
    check_dir(path_figs)

    #############################################
    # obter datas das imagens para plots
    season_dates = []
    [season_dates.append(os.path.basename(item)) for item in flist]
    season_dates = [pd.to_datetime(item.split('_')[1]).date() for item in season_dates]
    print(len(season_dates))
    #############################################

    ##############################################################
    ## Descomentar o bloco abaixo se quiser exportar figura dos clusters
    ##############################################################
    # Plotar e salvar curvas de cada cluster
    if(plot_curves == True):
        for k in range(k_cluster):
            print('k=',k)
            time_ini = datetime.now()
        
            
            series=X_origin[kmeans.labels_==k,:].T
            print(series.shape)
    
            # # Nome dos arquivos png de plots
            path_file=os.path.join(path_figs,'00_filter_sample_kmeans_'+str(k)+'.png')
            print(path_file)
            if True or not os.path.exists(path_file):
                plot_n_series(series,season_dates,'NDVI','Cluster Kmeans '+str(k)+' | '+str(series.shape[1])+' samples',
                              filename=path_file,
                              show_mean_std=True,
                             mean_color=k2color[k],
                              pltshow=pltshow
                             )
            time_end = datetime.now()
            print('elapsed time plots= ',time_end - time_ini)
            print('==========================================')


        
    ###############################
    # # Salvar GEOTIFF - raster clusterizado
    ###############################
    # cluster_raster = np.full(arr.shape[0:2], -1, dtype=np.int32)
    cluster_raster = np.ones(arr.shape[0:2], dtype=np.int8) * -99.
    cluster_raster[idx] = labels
    
    # Converter em numpy array
    cluster_raster=np.array(cluster_raster)

    # Plotar raster de cluster
    mapname=os.path.join(path_figs,'cluster_raster.png')
    plot_classified_raster(cluster_raster, gt, k2color,
                           title=f'Kmeans k={k_cluster}',nodata=-99,figname=mapname)

    # Exportar arquivo
    outname_raster = os.path.join(path_figs,f"raster_cluster_kmeans_k{k_cluster}_{feature}_{year}.tif")
    print(outname_raster)
    array_2_raster(outname_raster,cluster_raster,gt,proj, nodata=-99.)


    print('Results cluster',cluster_prop(cluster_raster,k_cluster,nodata=-99))
    print('Total elapsed time plots= ',datetime.now() - k_initime)
    print('=============================================') 

### Filtro passa baixa - mediana - exportar geojson da segmentacao

In [ ]:

wsize=3 # tamanho da janela movel
simplify_value = 0 # valor de simplify em metros (usar zero para nao alterar formas)

save_raster = True # se quiser salvar o geotiff filtrado, alem do geojson

outname_raster_filtered = outname_raster.replace('.tif',f'_filtered_{wsize}.tif')
outname_polygons_filtered = outname_raster.replace('.tif',f'_filtered_{wsize}_simple-{simplify_value}.geojson')

gdf_polygons, filt_raster = raster_to_polygons_with_simplify_fast(
    outname_raster, 
    size=wsize,
    simplify_tolerance_m=simplify_value,
    save_filtered_raster=save_raster,
    filtered_raster_path=outname_raster_filtered
)
gdf_polygons.to_file(outname_polygons_filtered, driver="GeoJSON")


# # Salvar png
plot_classified_raster(filt_raster, gt, k2color,
                       title=f'Kmeans k={k_cluster} (filtered) - Fmoda_{wsize}x{wsize}',nodata=-99.,figname=outname_raster_filtered.replace('.tif','.png'))


# CLUSTER 2 - DBSCAN

## Vantagem - nao necessita definir o numero de grupos desejado (bom para areas maiores com muitas dinamicas)
## desvantagens - 
#### * Pode atribuir pixels com valor -1, que corresponde a RUIDO
#### * tem parametro eps definido em funcao do dataset - temos funcao para estimar eps automatico, mas nem sempre funciona
#### * eps pode ser inputado manualmente - mas precisa testar valores (sugestao valores iniciar com 0.1, aumentar de 0.05)

### Funcao pra calculo de eps

In [ ]:
def estimate_eps_kdistance(X_scaled, min_samples, plot_path=None):
    """
    Estima eps para DBSCAN onde a derivada da curva K-distance normalizada ≈ 1.

    Args:
        X_scaled (np.array): Dados normalizados.
        min_samples (int): Número mínimo de vizinhos (DBSCAN).
        plot_path (str, optional): Caminho para salvar gráfico.

    Returns:
        float: eps sugerido.
    """
    from sklearn.neighbors import NearestNeighbors
    import matplotlib.pyplot as plt
    import numpy as np

    print("Calculando eps com base na derivada ≈ 1 da curva normalizada...")

    # Distância para o k-ésimo vizinho
    neighbors = NearestNeighbors(n_neighbors=min_samples)
    neighbors_fit = neighbors.fit(X_scaled)
    distances, _ = neighbors_fit.kneighbors(X_scaled)
    distances = np.sort(distances[:, min_samples - 1])

    # Normalizar eixo x e y
    x_norm = np.linspace(0, 1, len(distances))
    y_norm = (distances - distances.min()) / (distances.max() - distances.min())

    # Derivada da curva normalizada
    dy_dx = np.gradient(y_norm, x_norm)

    # Procurar primeiro ponto com derivada ≈ 1
    idx = np.where(np.isclose(dy_dx, 1.0, atol=0.05))[0]
    if len(idx) > 0:
        idx = idx[-1]
        criterio = "derivada ≈ 1"
    else:
        idx = np.argmax(dy_dx)
        criterio = "máximo da derivada"

    eps_sugerido = distances[idx]

    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(distances, label='K-distance curve')
    plt.plot(idx, distances[idx], 'ro', label=f'{criterio}: eps={eps_sugerido:.4f}')
    plt.xlabel('Pontos ordenados')
    plt.ylabel(f'Distância para o {min_samples}-ésimo vizinho')
    plt.title('Gráfico K-distance (com normalização para derivada)')
    plt.grid(True)
    plt.legend()

    if plot_path:
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"Gráfico salvo em: {plot_path}")
    else:
        plt.show()

    return eps_sugerido


#### executar DBSCAN

In [ ]:

# Data para restringir a serie temporal (start_date por padrao)
# date_slice = start_date

date_slice='2024-09-01'



In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import MinMaxScaler

k_initime = datetime.now()

#############################################################
# Definicao do parametro EPS automaticamente
# Se o eps definido automaticamente abaixo nao gerar resultado bom, alterar parametro abaixo para False

auto_eps = False

# Definicao manual de parametros
eps = .25       # Distância máxima entre vizinhos (variar conforme necessidade)
min_samples = 30 # Número mínimo de pontos por cluster


plot_curves = False # salvar png
pltshow = False # exibir no jupyter-notebook
#############################################################

for folder in folders[0:1]:
    print(folder)

    print('find folder =', os.path.join( folder,
                                        feature + '_hants_' + year,
                                        'reconstructed_' + str(freq_recon) + 'freq', '*.tif'))
    flist = sorted(glob.glob(os.path.join( folder, feature + '_hants_' + year,
                                          'reconstructed_' + str(freq_recon) + 'freq', '*.tif')))
    print(len(flist))

    flist = filter_images_by_date(flist, date_slice)
    print(os.path.basename(flist[0]))
    print(os.path.basename(flist[-1]))

    # Importar série temporal NDVI
    arr, gt, proj = stack_tifs_as_arrays(flist)

    ########################## 
    # Cluster
    ########################## 

    # Criar máscara de pixels válidos (média maior que zero)
    arr_mask = np.mean(arr, axis=2) > 0
    idx = np.where(arr_mask > 0)

    # Extrair apenas os pixels válidos
    X_origin = arr[arr_mask]

    # Normalização Min-Max por pixel (cada série de tempo é reescalada)
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X_origin)
    print(X_scaled.shape)

    ##############################################################
    # Clusterização com DBSCAN
    ##############################################################
    time_ini = datetime.now()


    # Pasta para salvar figuras dos clusters
    path_figs = os.path.join(folder, f'{feature}_hants{freq_recon}_{year}_figures_cluster_DBSCAN')
    check_dir(path_figs)
    
    # # Antes de clusterizar:
    if(auto_eps==True):
        eps = estimate_eps_kdistance(X_scaled, min_samples, 
                                     plot_path=os.path.join(path_figs, 'k_distance_plot.png'))
        print(eps)
        eps=eps/10.
        print(f"EPS definido automaticamente: {eps}")
    else: 
        print(f"EPS definido manualmente: {eps}")
    dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='euclidean')
    labels = dbscan.fit_predict(X_scaled)
    time_end = datetime.now()

    print(f"DBSCAN finalizado em: {time_end - time_ini}")
    print(f"Número de clusters encontrados (sem contar ruído): {len(set(labels)) - (1 if -1 in labels else 0)}")
    ##############################################################

    # Gerar escala de cores
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    k2color = generate_color_dict(n_clusters, cmap_name='viridis')
    k2color[-1] = (0.6, 0.6, 0.6)  # cinza para ruído



    # Datas das imagens
    season_dates = [pd.to_datetime(os.path.basename(f).split('_')[1]).date() for f in flist]
    print(len(season_dates))

    unique_clusters = set(labels)
    print("unique_clusters =", unique_clusters)

    
    for k in unique_clusters:
        time_ini = datetime.now()
        series = X_origin[labels == k, :].T
        print('k =', k, '||', series.shape)


        if(plot_curves == True):
            ##############################################################
            ## Descomentar o bloco abaixo se quiser exportar figura dos clusters
            ##############################################################
            path_file = os.path.join(path_figs, f'00_filter_sample_dbscan_k{k}.png')
            if True or not os.path.exists(path_file):
                plot_n_series(series, season_dates, 'NDVI',
                              f'Cluster DBSCAN #{k} | {series.shape[1]} samples',
                              filename=path_file,
                              show_mean_std=True,
                              mean_color=k2color[k],
                             pltshow=pltshow)
            print('elapsed time plots = ', datetime.now() - time_ini)

    ###############################
    # Retornar raster clusterizado
    ###############################
    cluster_raster = np.full(arr.shape[0:2], -99, dtype=np.int16)
    cluster_raster[idx] = labels

    # Plotar raster
    mapname = os.path.join(path_figs, f'raster_cluster_DBSCAN.png')
    plot_classified_raster(cluster_raster, gt, k2color,
                           title=f'Cluster raster - DBSCAN - eps={eps}, min_samples={min_samples}',
                           nodata=-99., figname=mapname)

    # Exportar
    outname_raster = os.path.join(path_figs, f"raster_cluster_DBSCAN_{feature}_{year}.tif")
    print(outname_raster)
    array_2_raster(outname_raster, cluster_raster, gt, proj, nodata=-99.)

    print("Valores únicos no raster:", np.unique(cluster_raster))
    print('Results cluster', cluster_prop(cluster_raster, n_clusters, nodata=-99))
    print('Total elapsed time plots =', datetime.now() - k_initime)
    print('=============================================')


#### Filtro passa baixa

In [ ]:
(print(cluster_raster.dtype))

In [ ]:

wsize = 3
simplify_value = 0
save_raster = False # se quiser salvar o geotiff filtrado, alem do geojson

outname_raster_filtered = outname_raster.replace('.tif',f'_filtered_{wsize}.tif')
outname_polygons_filtered = outname_raster.replace('.tif',f'_filtered_{wsize}_simple-{simplify_value}.geojson')

gdf_polygons, filt_raster = raster_to_polygons_with_simplify_fast(
    outname_raster, 
    size=wsize,
    simplify_tolerance_m=simplify_value,
    save_filtered_raster=save_raster,
    filtered_raster_path=outname_raster_filtered
)
gdf_polygons.to_file(outname_polygons_filtered, driver="GeoJSON")


# # Salvar png
plot_classified_raster(filt_raster, gt, k2color,
                       title=f'Kmeans k={k_cluster} (filtered) - Fmoda_{wsize}x{wsize}',nodata=-99.,figname=outname_raster_filtered.replace('.tif','.png'))


# CLUSTER 3 - HDBSCAN

## Variacao do DBSCAN

### Vantagem - nao necessida inputar numero de grupos desejado
#### Desvantagens
#### * Pode atribuir pixels com valor -1, que corresponde a RUIDO
#### * necessita de parametros min_cluster_size = dimensao do cluster (valor sugerido ==80)
#### * necessita de parametros min_samples = numero minimo de pontos por cluster - sugerido 30 para areas pequenas, para areas maiores, aumentar valor para nao gerar numero alto de clusters



### Sobre valores de "min_cluster_size"
#### [30-50] Areas homogeneas, cultura unica por talhao - maior numero de grupos
#### [60-100] Areas com variacoes moderadas entre talhoes da mesma cultura (recomendavel para laudos - valor 80)
#### [100-200]Areas muito diversas culturas diferentes proximas
#### Recomendado - Usar 80> como base, ajustar conforme necessidade


In [ ]:
import hdbscan


k_initime = datetime.now()

freq_recon= 4

plot_curves = False
pltshow = False
#############################################################
date_slice = '2024-09-01'

# Parametro de entrada do HDBSCAN
min_cluster_size = 100

min_samples = 5 #int(min_cluster_size/20)
#############################################################


In [ ]:
for folder in folders[0:1]:
    print(folder)

    print('find folder =', os.path.join( folder,
                                        feature + '_hants_' + year,
                                        'reconstructed_' + str(freq_recon) + 'freq', '*.tif'))
    flist = sorted(glob.glob(os.path.join( folder, feature + '_hants_' + year,
                                          'reconstructed_' + str(freq_recon) + 'freq', '*.tif')))
    print(len(flist))

    flist = filter_images_by_date(flist, date_slice)
    print(os.path.basename(flist[0]))
    print(os.path.basename(flist[-1]))

    # Importar série temporal NDVI
    arr, gt, proj = stack_tifs_as_arrays(flist)

    ########################## 
    # Cluster
    ########################## 
    ###############################
    # Cria máscara de pixels válidos (média maior que zero)

    arr_mask = np.mean(arr, axis=2) > 0
    idx = np.where(arr_mask >0) # mascara de pixels de interesse
    
    # Extrai apenas os pixels válidos
    X_origin = arr[arr_mask]
    
    # Normalização Min-Max por pixel (cada série de tempo é reescalada)
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X_origin)
    ##############################################################
    ##############################################################
    ##############################################################
    # Clusterização com HDBSCAN
    ##############################################################
    
    time_ini = datetime.now()
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size,
                                min_samples=min_samples,
                                metric='euclidean')  # você pode ajustar min_cluster_size
    labels = clusterer.fit_predict(X_scaled)
    time_end = datetime.now()

    print(f"HDBSCAN finalizado em: {time_end - time_ini}")
    print(f"Número de clusters encontrados (sem contar ruído): {len(set(labels)) - (1 if -1 in labels else 0)}")    
    ##############################################################
    ##############################################################
    ##############################################################
    
    # gerar escala de cores
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    k2color = generate_color_dict(n_clusters, cmap_name='viridis')
    k2color[-1] = (0.6, 0.6, 0.6)  # cor cinza para ruído


    
    ##########################################xxxxxxxxxxxxxxx


    ######################
    # Opcional - Gerar pasta para salvar figuras dos clusters
    path_figs=os.path.join(folder,f'{feature}_hants{freq_recon}_{year}_figures_cluster_HDBSCAN')
    check_dir(path_figs)

    #########################
    # obter datas das imagens para plots
    season_dates = []
    [season_dates.append(os.path.basename(item)) for item in flist]
    season_dates = [pd.to_datetime(item.split('_')[1]).date() for item in season_dates]
    print(len(season_dates))
    #############################################
    # unique_clusters = (set(labels) - {-1})  # exclui ruído
    unique_clusters =  (set(labels))
    print("unique_clusters=",unique_clusters)

    
    for k in unique_clusters:
        time_ini = datetime.now()    # contar tempo dos plots 
        
        series = X_origin[labels == k, :].T
        print('k=',k,'||',series.shape)
        # # Nome dos arquivos png de plots
        path_file=os.path.join(path_figs,'00_filter_sample_hdbscan_k'+str(k)+'.png')
        print(path_file)
        if True or not os.path.exists(path_file):
            plot_n_series(series,season_dates,'NDVI','Cluster HDBSCAN #'+str(k)+' | '+str(series.shape[1])+' samples',
                          filename=path_file,
                          show_mean_std=True,
                         mean_color=k2color[k],
                         pltshow=pltshow)
        time_end = datetime.now()
        print('elapsed time plots= ',time_end - time_ini)
        print('==========================================')


        
    ###############################
    # #Retornar raster clusterizado
    ###############################
    cluster_raster = np.full(arr.shape[0:2], -99, dtype=np.int16)
    # cluster_raster = np.ones(arr.shape[0:2], dtype=np.int16) * -99.
    cluster_raster[idx] = labels
    
    # Converter em numpy array
    cluster_raster=np.array(cluster_raster)

    # Plotar raster de cluster
    mapname=os.path.join(path_figs,f'raster_cluster_HDBSCAN_{feature}_{year}.png')
    plot_classified_raster(cluster_raster, gt, k2color,
                           title=f'Cluster raster - HDBSCAN - min_cluster_size={min_cluster_size}',nodata=-99.,figname=mapname)


    # Exportar arquivo
    outname_raster = os.path.join(path_figs,f"raster_cluster_HDBSCAN_{feature}_{year}.tif")
    print(outname_raster)
    array_2_raster(outname_raster,cluster_raster,gt,proj, nodata=-99.)


    
    print("Valores únicos no raster:", np.unique(cluster_raster))
    print('Results cluster',cluster_prop(cluster_raster,n_clusters,nodata=-99))
    print('Total elapsed time plots= ',datetime.now() - k_initime)
    print('=============================================') 

## Filtro mediana - HDBSCAN

In [ ]:


# Janela de filtro passa baixa - mediana
wsize=3



simplify_value = 0

save_raster = False # se quiser salvar o geotiff filtrado, alem do geojson

outname_raster_filtered = outname_raster.replace('.tif',f'_filtered_{wsize}.tif')
outname_polygons_filtered = outname_raster.replace('.tif',f'_filtered_{wsize}_simple-{simplify_value}.geojson')

gdf_polygons, filt_raster = raster_to_polygons_with_simplify_fast(
    outname_raster, 
    size=wsize,
    simplify_tolerance_m=simplify_value,
    save_filtered_raster=save_raster,
    filtered_raster_path=outname_raster_filtered
)
gdf_polygons.to_file(outname_polygons_filtered, driver="GeoJSON")


# # Salvar png
plot_classified_raster(filt_raster, gt, k2color,
                       title=f'HDBSCAN  (filtered) - Mediana_{wsize}x{wsize}',nodata=-99.,figname=outname_raster_filtered.replace('.tif','.png'))


# Extrair datas de plantio, maximo e senescencia

In [ ]:
## Vai usar a clusterizacao rodada por ultimo - temos que ajustar isso

In [ ]:
###

# Versao 2 - datas de plantio

## Aqui usamos a serie temporal de ndvi, datas da safra, e o geojson resultante da clusterizacao

In [ ]:

from scipy.signal import argrelextrema

def identificar_pontos_criticos_com_k(ndvi_suavizado, datas,
                                      k=2,
                                      order=3,
                                      ndvi_min_threshold=1000,
                                      ndvi_max_threshold=5000, # valor maximo de ndvi onde ocorre um minimo local
                                      plotar=True):
    """
    Identifica pontos críticos (mínimos, máximos, inflexões) em uma série NDVI suavizada.
    Também estima datas de plantio com base em mínimos anteriores aos máximos.

    Parâmetros:
    - ndvi_suavizado: lista ou array com valores de NDVI
    - datas: lista de strings com datas no formato 'YYYY-MM-DD'
    - k: intervalo mínimo (em índices) entre mínimo e máximo para considerar um ciclo
    - order: parâmetro para suavização de extremos (usado por argrelextrema)
    - ndvi_min_threshold: valor mínimo de NDVI para considerar início de ciclo
    - ndvi_max_threshold: valor mínimo de NDVI para considerar pico vegetativo
    - plotar: se True, plota os resultados

    Retorna:
    - DataFrame com pontos críticos e tipos
    """
    datas_dt = [datetime.strptime(str(d), '%Y-%m-%d') for d in datas]
    ndvi = np.array(ndvi_suavizado)

    # Detectar máximos, mínimos e inflexões
    idx_max = argrelextrema(ndvi, np.greater, order=order)[0]
    idx_min = argrelextrema(ndvi, np.less, order=order)[0]
    idx_infl = argrelextrema(np.gradient(ndvi), np.less, order=order)[0]

    pontos = []

    for i in idx_max:
        if ndvi[i] > ndvi_max_threshold:
            pontos.append((datas_dt[i], ndvi[i], f'max_{i}'))

    for i in idx_min:
        if ndvi[i] < ndvi_max_threshold:
            pontos.append((datas_dt[i], ndvi[i], f'min_{i}'))

    for i in idx_infl:
        pontos.append((datas_dt[i], ndvi[i], f'inflex_{i}'))

    pontos = sorted(pontos, key=lambda x: x[0])
    df_pontos = pd.DataFrame(pontos, columns=['Data', 'NDVI', 'Tipo_Ponto'])

    # Estimar data de plantio com base em mínimo anterior ao máximo
    estimativas = []
    for i, row in df_pontos[df_pontos['Tipo_Ponto'].str.startswith('max')].iterrows():
        idx = datas_dt.index(row['Data'])
        for j in range(idx - 1, max(0, idx - k - 5), -1):
            if ndvi[j] < ndvi_min_threshold:
                estimativas.append((datas_dt[j], ndvi[j], 'estimado_plantio'))
                break

    for est in estimativas:
        df_pontos.loc[len(df_pontos)] = est

    df_pontos = df_pontos.sort_values('Data').reset_index(drop=True)

    if plotar:
        plt.figure(figsize=(12, 5))
        plt.plot(datas_dt, ndvi, label='NDVI')
        for i, row in df_pontos.iterrows():
            plt.scatter(row['Data'], row['NDVI'],
                        label=row['Tipo_Ponto'], s=60,
                        marker='x' if 'estimado_plantio' in row['Tipo_Ponto'] else 'o')
        plt.legend()
        plt.grid(True)
        plt.title(f"Detecção de Pontos Críticos - cluster {k}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.ylim([0,10000])
        plt.show()

    return df_pontos


In [ ]:
def identificar_ciclos_agricolas(df_pontos, ndvi, datas,
                                 ndvi_max_threshold=5000, # valor maximo de ndvi onde ocorre um minimo local
                                 delta_ndvi_threshold=500,
                                 queda_percentual_minima=0.2):
    """
    Identifica ciclos agrícolas com base em pontos críticos do NDVI.
    Considera como colheita apenas um mínimo local após o pico,
    desde que a queda no NDVI seja significativa.
    """
    from datetime import datetime

    datas_dt = [datetime.strptime(d, '%Y-%m-%d') for d in datas]
    ndvi = np.array(ndvi)

    # Converter coluna 'Data' para datetime, se necessário
    if df_pontos['Data'].dtype == 'O':
        df_pontos['Data'] = pd.to_datetime(df_pontos['Data'])

    minimos = df_pontos[df_pontos['Tipo_Ponto'].str.startswith('min')].sort_values('Data')
    maximos = df_pontos[
        (df_pontos['Tipo_Ponto'].str.startswith('max')) &
        (df_pontos['NDVI'] > ndvi_max_threshold)
    ].sort_values('Data')

    ciclos = []

    for _, max_row in maximos.iterrows():
        data_max = max_row['Data']
        idx_max = datas_dt.index(data_max)
        ndvi_max = ndvi[idx_max]

        # Busca o mínimo anterior como data de plantio
        anteriores = minimos[minimos['Data'] < data_max]
        if anteriores.empty:
            continue
        data_plantio = anteriores.iloc[-1]['Data']

        # Busca o próximo mínimo após o pico
        posteriores = minimos[minimos['Data'] > data_max]

        colheita = None
        em_andamento = True

        if not posteriores.empty:
            min_post = posteriores.iloc[0]
            data_min_post = min_post['Data']
            idx_min_post = datas_dt.index(data_min_post)
            ndvi_min_post = ndvi[idx_min_post]
#
            delta = ndvi_max - ndvi_min_post
            queda_relativa = delta / ndvi_max if ndvi_max > 0 else 0

            if delta > delta_ndvi_threshold or queda_relativa > queda_percentual_minima:
                colheita = data_min_post
                em_andamento = False
            # caso contrário, não define colheita

        # Se não houver mínimo posterior, ciclo segue em andamento
        ciclo = {
            'plantio': data_plantio,
            'pico': data_max,
            'NDVI_max': ndvi_max,
            'colheita': colheita,
            'em_andamento': em_andamento
        }
        ciclos.append(ciclo)

    return ciclos



def formatar_data(d):
    """Garante que a data esteja no formato 'YYYY-MM-DD' como string."""
    if d is None or pd.isna(d):
        return None
    if isinstance(d, (pd.Timestamp, date)):
        return d.strftime('%Y-%m-%d')
    return str(d)

# Merge da segmentacao de dinamicas com datas de plantio e colheita




In [ ]:
# aqui estamos igualando a serie de datas ao "date_slice" utilizado na clusterizacao
for folder in folders[0:1]:
    flist = sorted(glob.glob(os.path.join( folder, feature + '_hants_' + year,
                                              'reconstructed_' + str(freq_recon) + 'freq', '*.tif')))
    flist = filter_images_by_date(flist, date_slice)
    # obter datas das imagens para plots
    season_dates = []
    [season_dates.append(os.path.basename(item)) for item in flist]
    season_dates = [pd.to_datetime(item.split('_')[1]).date() for item in season_dates]
    season_dates = [str(item) for item in season_dates]
    print(len(season_dates))
    print(season_dates)


In [ ]:
dados_cluster = []


# Iterar por cada cluster identificado e avaliar a serie temporal de ndvi - identificar data de plantio, pico e colheita
for k in unique_clusters:
    series = X_origin[labels == k, :].T
    ndvi = list(np.mean(series, axis=1))

    # 1. Detectar pontos críticos
    df_pontos = identificar_pontos_criticos_com_k(ndvi_suavizado=ndvi, datas=datas, k=k)

    # 2. Identificar ciclos agrícolas
    ciclos = identificar_ciclos_agricolas(df_pontos, ndvi, datas)

    # Formatar cada ciclo como dicionário com datas como string
    ciclos_dict = []
    for ciclo in ciclos:
        ciclos_dict.append({
            'plantio': formatar_data(ciclo.get('plantio')),
            'pico': formatar_data(ciclo.get('pico')),
            'colheita': formatar_data(ciclo.get('colheita'))
        })

    # Extrair último ciclo
    if ciclos_dict:
        ultimo_ciclo = ciclos_dict[-1]
        data_plantio = ultimo_ciclo['plantio']
        data_colheita = ultimo_ciclo['colheita']
        data_pico = ultimo_ciclo['pico']
        em_andamento = data_colheita is None
    else:
        data_plantio = None
        data_colheita = None
        data_pico = None
        em_andamento = False

    dados_cluster.append({
        'label': k,
        'ciclos': ciclos_dict,
        'plantio': data_plantio,
        'colheita': data_colheita,
        'pico': data_pico,
        'em_andamento': str(em_andamento),  # mantém como string ("True"/"False")
        'ndvi_serie': ndvi,
        'datas_serie': datas
    })

# Converter em DataFrame
df_ciclos_clusters = pd.DataFrame(dados_cluster)

# Merge com GeoDataFrame de segmentação
gdf_polygons_ciclos = gdf_polygons.merge(df_ciclos_clusters, left_on='value', right_on='label', how='left')

# Calcular areas de poligonos
gdf_polygons_ciclos = calcular_area_ha(gdf_polygons_ciclos)

# Reordenar geodataframe por label de cluster e area
gdf_polygons_ciclos = gdf_polygons_ciclos.sort_values(by=['label','area_ha'], ascending = [True,False])

# Converter colunas para string
gdf_polygons_ciclos['plantio'] = gdf_polygons_ciclos['plantio'].astype(str)
gdf_polygons_ciclos['colheita'] = gdf_polygons_ciclos['colheita'].astype(str)
gdf_polygons_ciclos['pico'] = gdf_polygons_ciclos['pico'].astype(str)
gdf_polygons_ciclos['em_andamento'] = gdf_polygons_ciclos['em_andamento'].astype(str)

display(gdf_polygons_ciclos)

gdf_polygons_ciclos.label = gdf_polygons_ciclos.label.astype(str)
fig,ax = plt.subplots(figsize = [7,7])
gdf_polygons_ciclos.plot(ax=ax,column='label',legend=True, cmap='viridis')
plt.show()

# Exportar Geojson
outname_final = os.path.join(folder, 'Result_polygons_with_dates.geojson')

if(os.path.isfile(outname_final)):
   os.remove(outname_final) 

gdf_polygons_ciclos.to_file(outname_final,driver='GeoJSON')

In [ ]:
display(gdf_polygons_ciclos)

In [ ]:
print(ciclos_dict)